# NeoOLAF × RAGTree — Parallel-5 with PROCESS isolation

This replaces the unsafe thread-level document parallelism.

## Why

EventStoryLine `v1.6/v1.7` temporarily monkey-patches module-level pipeline functions while a document runs. That execution shell is safe for one document at a time **inside one Python process**, but it is not thread-safe across several documents in the same interpreter.

Therefore this notebook launches **five independent Python processes**. Each process has:
- its own Python interpreter;
- its own NeoOLAF/module globals;
- its own EventStoryLine monkey-patch lifecycle;
- its own run directory;
- its own stdout/stderr log.

This preserves the requested parallelism without cross-document interpreter state collisions.

## Parallelism

- EventStoryLine: **5 documents in parallel**, **4 layer workers per document**.
- FinCausal: **5 documents in parallel**, **1 layer worker per document**.
- Dataset phases remain sequential: EventStoryLine first; FinCausal starts only after all five EventStoryLine processes succeed.

Use a **fresh kernel** for this notebook. Do not continue the old ThreadPoolExecutor run.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, subprocess, textwrap, re, traceback
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate

RUN_PAID = True
RUN_ORDER = ["eventstoryline", "fincausal"]
MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

DOCUMENT_WORKERS = {
    "eventstoryline": 5,
    "fincausal": 5,
}
LAYER_WORKERS = {
    "eventstoryline": 4,
    "fincausal": 1,
}
EXPECTED_VERSIONS = {
    "eventstoryline": "v1.7",
    "fincausal": "unified-v1.3.1-selection-hotfix",
}

assert DOCUMENT_WORKERS["eventstoryline"] == 5
assert DOCUMENT_WORKERS["fincausal"] == 5
assert LAYER_WORKERS["eventstoryline"] == 4
assert LAYER_WORKERS["fincausal"] == 1

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python executable used for child processes:", sys.executable)
print("EventStoryLine: 5 processes × 4 layer workers")
print("FinCausal     : 5 processes × 1 layer worker")


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Python executable used for child processes: c:\Users\galencarmedeiro\NeoOLAF\.venv\Scripts\python.exe
EventStoryLine: 5 processes × 4 layer workers
FinCausal     : 5 processes × 1 layer worker


## Exact frozen five-record selection

Both datasets are resolved from the already persisted `smoke5_record_keys`, avoiding ambiguous `document_id` values.


In [2]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

dataset_rows = {
    k: expstate.read_jsonl(DATASET_FILES[k])
    for k in RUN_ORDER
}

STATE_DIR = EXPERIMENT_ROOT / "state"
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
dev_manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)

for k in RUN_ORDER:
    e = dev_manifest[k]
    assert e.get("one_doc_completed"), f"{k}: one-doc incomplete"
    assert e.get("smoke5_already_run"), f"{k}: smoke-5 incomplete"
    assert e.get("best_version") == EXPECTED_VERSIONS[k], (
        k, e.get("best_version"), EXPECTED_VERSIONS[k]
    )

def resolve_exact_smoke_rows(dataset_key, rows):
    keys = list(dev_manifest[dataset_key].get("smoke5_record_keys") or [])
    assert len(keys) == 5 and len(set(keys)) == 5, (dataset_key, keys)

    by_key = {}
    for r in rows:
        rk = expstate.record_key(dataset_key, r)
        by_key.setdefault(rk, []).append(r)

    resolved = []
    for rk in keys:
        matches = by_key.get(rk, [])
        assert len(matches) == 1, (dataset_key, rk, len(matches))
        resolved.append(matches[0])

    assert [expstate.record_key(dataset_key, r) for r in resolved] == keys
    return resolved

fixed_rows = {
    k: resolve_exact_smoke_rows(k, dataset_rows[k])
    for k in RUN_ORDER
}

# Gold-isolation audit, controller-side only.
for k in RUN_ORDER:
    for r in fixed_rows[k]:
        clean = expstate.strip_gold(r)
        forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean)
        assert not forbidden, (k, forbidden)

# Confirm corrected FinCausal smoke is the positive five.
for r in fixed_rows["fincausal"]:
    c = expstate.gold_contract_summary("fincausal", r)
    assert c["gold_target_relation_count"] == 1, c
    assert c["gold_entity_count"] >= 2, c

print("EventStoryLine frozen keys:")
for r in fixed_rows["eventstoryline"]:
    print(" -", expstate.record_key("eventstoryline", r), "|", r.get("document_id"), "|", r.get("title"))

print("\nFinCausal frozen keys:")
for r in fixed_rows["fincausal"]:
    print(" -", expstate.record_key("fincausal", r), "|", r.get("title"))

print("\nGold isolation: OK")
print("No API call has been made.")


EventStoryLine frozen keys:
 - eventstoryline:0:9476bcab3bb12b52 | EventStoryLine - 1_10ecbplus | 1_10ecbplus
 - eventstoryline:1:e262639faff6efc3 | EventStoryLine - 1_11ecbplus | 1_11ecbplus
 - eventstoryline:2:ff2644df5186a9c2 | EventStoryLine - 1_12ecbplus | 1_12ecbplus
 - eventstoryline:3:529aeeecab91ee4b | EventStoryLine - 1_13ecbplus | 1_13ecbplus
 - eventstoryline:4:a6e7f686d2b2cc24 | EventStoryLine - 1_14ecbplus | 1_14ecbplus

FinCausal frozen keys:
 - fincausal:10:7551d279863a4a5c | 0001.00005.1
 - fincausal:11:ffe2d4714fd469d5 | 0001.00005.2
 - fincausal:12:4e8690e508dcacd0 | 0001.00007
 - fincausal:13:290790a5cf066893 | 0001.00009
 - fincausal:14:6693dc85e8637424 | 0001.00011

Gold isolation: OK
No API call has been made.


## Create isolated worker script and run state

The worker script is generated under a new run root and invoked via the same Python executable as this notebook kernel.

The previous unsafe thread-run directory is **not reused**.


In [3]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "parallel5_process_isolated_v1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

WORKER_SCRIPT = RUNS_ROOT / "_parallel5_worker.py"
PROGRESS_PATH = RUNS_ROOT / "parallel5_progress.json"
SUMMARY_PATH = RUNS_ROOT / "parallel5_summary.json"
USAGE_WINDOW_PATH = RUNS_ROOT / "openrouter_usage_window.json"

worker_code = r"""
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import argparse, os, sys, json, time, shutil, re, traceback

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

parser = argparse.ArgumentParser()
parser.add_argument("--project-root", required=True)
parser.add_argument("--dataset", required=True, choices=["eventstoryline", "fincausal"])
parser.add_argument("--record-key", required=True)
parser.add_argument("--run-root", required=True)
parser.add_argument("--layer-workers", required=True, type=int)
parser.add_argument("--model", required=True)
parser.add_argument("--host", required=True)
parser.add_argument("--reasoning-effort", default="minimal")
parser.add_argument("--max-tokens", type=int, default=8192)
parser.add_argument("--request-timeout", type=int, default=180)
args = parser.parse_args()

PROJECT_ROOT = Path(args.project_root).resolve()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_8 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

dataset_key = args.dataset
rkey = args.record_key
run_root = Path(args.run_root).resolve()
run_dir = run_root / dataset_key / "parallel5" / safe_dir_name(rkey)

failure_path = run_dir / "worker_failure.json"

try:
    RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
    PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
    ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)
    DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)
    RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)

    rows = expstate.read_jsonl(DATASET_FILES[dataset_key])
    matches = [r for r in rows if expstate.record_key(dataset_key, r) == rkey]
    if len(matches) != 1:
        raise RuntimeError(f"{dataset_key} {rkey}: expected 1 normalized row, got {len(matches)}")
    gold_record = matches[0]

    configs = {
        "eventstoryline": {
            "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
            "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
            "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
            "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
            "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
            "version": "v1.7",
        },
        "fincausal": {
            "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
            "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
            "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
            "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
            "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
            "version": "unified-v1.3.1-selection-hotfix",
        },
    }
    cfg = configs[dataset_key]
    ontology_path = RAW_ONTOLOGY_FILES[dataset_key]

    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    pre_gold_contract = expstate.gold_contract_summary(dataset_key, gold_record)
    clean_record = expstate.strip_gold(gold_record)
    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean_record)
    if forbidden:
        raise RuntimeError(f"Gold leakage in worker input: {forbidden}")

    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    if gold_path.exists():
        gold_path.unlink()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    started = utc_now()
    t0 = time.perf_counter()

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ontology_path,
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=args.model,
            api_key=api_key,
            host=args.host,
            workers=args.layer_workers,
            max_tokens=args.max_tokens,
            request_timeout=args.request_timeout,
            reasoning_effort=args.reasoning_effort,
            verbose=True,
            clean_run_dir=False,
        )

        # Gold becomes visible only after Layer 12 returned.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        summary = esl_v17.analyze_run(
            run_dir=run_dir,
            gold_jsonl=gold_path,
            catalog_path=cfg["catalog"],
            aliases_path=cfg["aliases"],
        )
        result = {
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "relation_metrics": (
                summary.get("projected_relation_evaluation")
                or summary.get("strict_relation_evaluation")
                or {}
            ),
            "endpoint_metrics": (
                summary.get("relation_endpoint_evaluation")
                or summary.get("event_entity_evaluation")
                or {}
            ),
            "candidate_pool": (
                summary.get("candidate_pool_coverage")
                or summary.get("candidate_pool")
                or {}
            ),
            "pre_run_gold_contract": pre_gold_contract,
        }

    else:
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key,
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ontology_path,
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=args.model,
            api_key=api_key,
            host=args.host,
            workers=args.layer_workers,
            max_tokens=args.max_tokens,
            request_timeout=args.request_timeout,
            reasoning_effort=args.reasoning_effort,
            verbose=True,
            clean_run_dir=False,
        )

        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "pre_run_gold_contract": pre_gold_contract,
        })

        expected_gold = int(pre_gold_contract["gold_target_relation_count"])
        evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
        if expected_gold > 0 and evaluated_gold == 0:
            raise RuntimeError(
                f"FinCausal evaluator integrity error: expected {expected_gold} gold relation(s), evaluator saw 0."
            )

    result.update({
        "run_dir": str(run_dir),
        "started_at_utc": started,
        "finished_at_utc": utc_now(),
        "elapsed_seconds": time.perf_counter() - t0,
        "model": args.model,
        "layer_workers": args.layer_workers,
        "process_id": os.getpid(),
        "gold_visible_to_pipeline": False,
    })

    adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    print("WORKER_SUCCESS", dataset_key, rkey, json.dumps(result.get("relation_metrics") or {}))
    sys.exit(0)

except Exception as exc:
    try:
        run_dir.mkdir(parents=True, exist_ok=True)
        payload = {
            "dataset": dataset_key,
            "record_key": rkey,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
            "failed_at_utc": utc_now(),
            "process_id": os.getpid(),
        }
        failure_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    except Exception:
        pass
    traceback.print_exc()
    sys.exit(1)
"""

WORKER_SCRIPT.write_text(textwrap.dedent(worker_code), encoding="utf-8")

# Syntax-check the generated worker without executing it.
compile(WORKER_SCRIPT.read_text(encoding="utf-8"), str(WORKER_SCRIPT), "exec")

def atomic_json(path, obj):
    expstate.atomic_write_json(Path(path), obj)

def load_json(path, default=None):
    p = Path(path)
    if not p.exists():
        return default
    return json.loads(p.read_text(encoding="utf-8"))

def fresh_progress():
    return {
        "schema_version": 1,
        "experiment": "parallel5_process_isolated_v1",
        "created_at_utc": utc_now(),
        "experiment_started_at_utc": None,
        "experiment_finished_at_utc": None,
        "model": MODEL_NAME,
        "datasets": {
            k: {
                "version": EXPECTED_VERSIONS[k],
                "document_workers": DOCUMENT_WORKERS[k],
                "layer_workers": LAYER_WORKERS[k],
                "record_keys": [expstate.record_key(k, r) for r in fixed_rows[k]],
                "completed_record_keys": [],
                "failures": {},
                "started_at_utc": None,
                "finished_at_utc": None,
            }
            for k in RUN_ORDER
        },
    }

progress = load_json(PROGRESS_PATH, None)
if progress is None:
    progress = fresh_progress()
    atomic_json(PROGRESS_PATH, progress)
else:
    assert progress["model"] == MODEL_NAME
    for k in RUN_ORDER:
        assert progress["datasets"][k]["version"] == EXPECTED_VERSIONS[k]
        assert progress["datasets"][k]["record_keys"] == [
            expstate.record_key(k, r) for r in fixed_rows[k]
        ]
        assert progress["datasets"][k]["layer_workers"] == LAYER_WORKERS[k]

def write_usage_window():
    payload = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {
            k: {
                "version": progress["datasets"][k]["version"],
                "document_workers": progress["datasets"][k]["document_workers"],
                "layer_workers": progress["datasets"][k]["layer_workers"],
                "started_at_utc": progress["datasets"][k].get("started_at_utc"),
                "finished_at_utc": progress["datasets"][k].get("finished_at_utc"),
            }
            for k in RUN_ORDER
        },
    }
    atomic_json(USAGE_WINDOW_PATH, payload)
    return payload

write_usage_window()

print("RUNS_ROOT:", RUNS_ROOT)
print("Worker script:", WORKER_SCRIPT)
print("Worker syntax check: OK")
print("Fresh isolated run root: previous ThreadPool run is not reused.")


RUNS_ROOT: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\parallel5_process_isolated_v1
Worker script: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\parallel5_process_isolated_v1\_parallel5_worker.py
Worker syntax check: OK
Fresh isolated run root: previous ThreadPool run is not reused.


## Aggregation


In [4]:
def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def result_path(dataset_key, rkey):
    return RUNS_ROOT / dataset_key / "parallel5" / safe_dir_name(rkey) / "posthoc_evaluation.json"

def _count(m, *names):
    for name in names:
        if isinstance(m, dict) and m.get(name) is not None:
            return int(m.get(name) or 0)
    return 0

def normalized_counts(m):
    tp = _count(m, "tp", "true_positive")
    fp = _count(m, "fp", "false_positive")
    fn = _count(m, "fn", "false_negative")
    pred = _count(m, "pred", "predicted")
    gold = _count(m, "gold", "gold_unique")
    if pred == 0 and tp + fp:
        pred = tp + fp
    if gold == 0 and tp + fn:
        gold = tp + fn
    return pred, gold, tp, fp, fn

def aggregate(dataset_key):
    done = set(progress["datasets"][dataset_key].get("completed_record_keys") or [])
    rows = []
    for r in fixed_rows[dataset_key]:
        rk = expstate.record_key(dataset_key, r)
        if rk in done:
            p = result_path(dataset_key, rk)
            if not p.exists():
                raise RuntimeError(f"Missing result for completed key: {rk}")
            rows.append(json.loads(p.read_text(encoding="utf-8")))

    cs = [normalized_counts(r.get("relation_metrics") or {}) for r in rows]
    pred = sum(x[0] for x in cs)
    gold = sum(x[1] for x in cs)
    tp = sum(x[2] for x in cs)
    fp = sum(x[3] for x in cs)
    fn = sum(x[4] for x in cs)

    p = tp / (tp + fp) if tp + fp else 0.0
    rr = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2*p*rr/(p+rr) if p+rr else 0.0
    mean_doc_f1 = (
        sum(float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0) for r in rows) / len(rows)
        if rows else 0.0
    )

    return {
        "dataset": dataset_key,
        "version": EXPECTED_VERSIONS[dataset_key],
        "docs": len(rows),
        "pred": pred,
        "gold": gold,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": p,
        "recall": rr,
        "micro_f1": f1,
        "mean_doc_f1": mean_doc_f1,
        "document_processes": DOCUMENT_WORKERS[dataset_key],
        "layer_workers_per_process": LAYER_WORKERS[dataset_key],
    }

def save_summary():
    s = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "experiment_started_at_utc": progress.get("experiment_started_at_utc"),
        "experiment_finished_at_utc": progress.get("experiment_finished_at_utc"),
        "datasets": {k: aggregate(k) for k in RUN_ORDER},
    }
    atomic_json(SUMMARY_PATH, s)
    write_usage_window()
    return s

print("Aggregation helpers ready.")


Aggregation helpers ready.


## Execute — five OS processes in parallel

Child stdout/stderr goes to one log file per document, so Jupyter's own stdout cannot be closed or corrupted by a child pipeline.

The parent only prints process lifecycle status.


In [5]:
if not RUN_PAID:
    print("RUN_PAID=False -> stopped before API calls.")
else:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    if progress.get("experiment_started_at_utc") is None:
        progress["experiment_started_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        write_usage_window()

    for dataset_key in RUN_ORDER:
        ds = progress["datasets"][dataset_key]

        if dataset_key == "fincausal":
            if len(progress["datasets"]["eventstoryline"]["completed_record_keys"]) != 5:
                raise RuntimeError("FinCausal blocked until EventStoryLine is 5/5 successful.")

        completed = set(ds.get("completed_record_keys") or [])
        pending_rows = [
            r for r in fixed_rows[dataset_key]
            if expstate.record_key(dataset_key, r) not in completed
        ]

        if not pending_rows:
            print(f"\nSKIP {dataset_key}: already 5/5 complete.")
            continue

        if ds.get("started_at_utc") is None:
            ds["started_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()

        print(
            f"\n=== LAUNCH {dataset_key.upper()} ===\n"
            f"{len(pending_rows)} child processes simultaneously | "
            f"layer_workers={LAYER_WORKERS[dataset_key]}"
        )

        children = {}
        for r in pending_rows:
            rk = expstate.record_key(dataset_key, r)
            doc_dir = RUNS_ROOT / dataset_key / "parallel5" / safe_dir_name(rk)
            doc_dir.mkdir(parents=True, exist_ok=True)
            log_path = doc_dir / "worker_console.log"

            # Parent owns the log handle; the child receives an OS-level handle.
            log_handle = open(log_path, "w", encoding="utf-8", buffering=1)

            cmd = [
                sys.executable,
                str(WORKER_SCRIPT),
                "--project-root", str(PROJECT_ROOT),
                "--dataset", dataset_key,
                "--record-key", rk,
                "--run-root", str(RUNS_ROOT),
                "--layer-workers", str(LAYER_WORKERS[dataset_key]),
                "--model", MODEL_NAME,
                "--host", OPENROUTER_HOST,
                "--reasoning-effort", REASONING_EFFORT,
                "--max-tokens", str(MAX_TOKENS),
                "--request-timeout", str(REQUEST_TIMEOUT),
            ]

            env = os.environ.copy()
            env["NEOOLAF_PROJECT_ROOT"] = str(PROJECT_ROOT)

            proc = subprocess.Popen(
                cmd,
                stdout=log_handle,
                stderr=subprocess.STDOUT,
                env=env,
                cwd=str(PROJECT_ROOT),
            )
            children[rk] = {
                "proc": proc,
                "log_handle": log_handle,
                "log_path": log_path,
                "document_id": r.get("document_id"),
                "title": r.get("title"),
                "started_at": time.time(),
            }
            print(f"START pid={proc.pid} | {rk}")

        # Monitor all five.
        last_status_print = 0.0
        while children:
            now = time.time()
            finished = []

            for rk, info in list(children.items()):
                rc = info["proc"].poll()
                if rc is None:
                    continue

                info["log_handle"].close()
                finished.append(rk)

                if rc == 0 and result_path(dataset_key, rk).exists():
                    if rk not in ds["completed_record_keys"]:
                        ds["completed_record_keys"].append(rk)
                    ds.setdefault("failures", {}).pop(rk, None)
                    elapsed = time.time() - info["started_at"]
                    print(f"DONE pid={info['proc'].pid} | {rk} | wall={elapsed:.1f}s")
                else:
                    failure_file = (
                        RUNS_ROOT / dataset_key / "parallel5" /
                        safe_dir_name(rk) / "worker_failure.json"
                    )
                    failure = {
                        "record_key": rk,
                        "returncode": rc,
                        "log_path": str(info["log_path"]),
                        "failure_file": str(failure_file) if failure_file.exists() else None,
                    }
                    if failure_file.exists():
                        try:
                            failure["worker_failure"] = json.loads(
                                failure_file.read_text(encoding="utf-8")
                            )
                        except Exception:
                            pass
                    ds.setdefault("failures", {})[rk] = failure
                    print(
                        f"FAILED pid={info['proc'].pid} | {rk} | rc={rc}\n"
                        f"  log: {info['log_path']}"
                    )

                atomic_json(PROGRESS_PATH, progress)
                write_usage_window()

            for rk in finished:
                children.pop(rk, None)

            if children and now - last_status_print >= 15:
                print(
                    "Still running:",
                    ", ".join(
                        f"{rk}(pid={info['proc'].pid})"
                        for rk, info in children.items()
                    )
                )
                last_status_print = now

            if children:
                time.sleep(1.0)

        if len(ds["completed_record_keys"]) == 5:
            ds["finished_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
            write_usage_window()
            print(f"\n{dataset_key}: 5/5 SUCCESS")
            pprint(aggregate(dataset_key))
        else:
            atomic_json(PROGRESS_PATH, progress)
            save_summary()
            raise RuntimeError(
                f"{dataset_key}: only {len(ds['completed_record_keys'])}/5 succeeded. "
                "Inspect the per-document worker_console.log files. "
                "Rerunning this notebook will skip successful records and retry only failures."
            )

    if all(len(progress["datasets"][k]["completed_record_keys"]) == 5 for k in RUN_ORDER):
        progress["experiment_finished_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        write_usage_window()

    final_summary = save_summary()
    print("\nPROCESS-ISOLATED PARALLEL-5 COMPLETE")
    pprint(final_summary)
    print("\nSummary:", SUMMARY_PATH)
    print("Usage window:", USAGE_WINDOW_PATH)



=== LAUNCH EVENTSTORYLINE ===
5 child processes simultaneously | layer_workers=4
START pid=4700 | eventstoryline:0:9476bcab3bb12b52
START pid=2292 | eventstoryline:1:e262639faff6efc3
START pid=18572 | eventstoryline:2:ff2644df5186a9c2
START pid=22200 | eventstoryline:3:529aeeecab91ee4b
START pid=1552 | eventstoryline:4:a6e7f686d2b2cc24
Still running: eventstoryline:0:9476bcab3bb12b52(pid=4700), eventstoryline:1:e262639faff6efc3(pid=2292), eventstoryline:2:ff2644df5186a9c2(pid=18572), eventstoryline:3:529aeeecab91ee4b(pid=22200), eventstoryline:4:a6e7f686d2b2cc24(pid=1552)
FAILED pid=4700 | eventstoryline:0:9476bcab3bb12b52 | rc=1
  log: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\parallel5_process_isolated_v1\eventstoryline\parallel5\eventstoryline_0_9476bcab3bb12b52\worker_console.log
FAILED pid=2292 | eventstoryline:1:e262639faff6efc3 | rc=1
  log: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\parallel5_process_isolated_v1\eventstoryline\parallel5

RuntimeError: eventstoryline: only 0/5 succeeded. Inspect the per-document worker_console.log files. Rerunning this notebook will skip successful records and retry only failures.

## Final report — zero API calls

Send me the executed notebook or `parallel5_summary.json`. If a child fails, also send its `worker_console.log`.


In [ ]:
summary = save_summary()
usage = write_usage_window()

print("PROCESS-ISOLATED PARALLEL-5 RESULTS")
print("===================================")
for k in RUN_ORDER:
    a = summary["datasets"][k]
    print(
        f"\n{k}: docs={a['docs']}/5 | "
        f"processes={a['document_processes']} | "
        f"layer_workers/process={a['layer_workers_per_process']}"
    )
    print(
        f"P={a['precision']:.6f} "
        f"R={a['recall']:.6f} "
        f"micro-F1={a['micro_f1']:.6f} "
        f"mean-doc-F1={a['mean_doc_f1']:.6f} "
        f"TP={a['tp']} FP={a['fp']} FN={a['fn']}"
    )

print("\nOPENROUTER USAGE WINDOW")
pprint(usage)
print("\nGenerated:")
print(" -", PROGRESS_PATH)
print(" -", SUMMARY_PATH)
print(" -", USAGE_WINDOW_PATH)
